In [2]:
# @title
import pandas as pd

# =========================================================
# NEWSPAPER SELLER PROBLEM
# GOOD - FAIR - POOR NEWSDAY
# =========================================================

# ---------------------------------------------------------
# 1. TYPE OF NEWSDAY DISTRIBUTION
# ---------------------------------------------------------
newsday_types = ['Good', 'Fair', 'Poor']
newsday_probs = [0.35, 0.45, 0.20]

# ---------------------------------------------------------
# 2. DEMAND DISTRIBUTION
# ---------------------------------------------------------
demand_levels = [40, 60, 80]

good_probs = [0.25, 0.40, 0.35]
fair_probs = [0.20, 0.45, 0.35]
poor_probs = [0.40, 0.35, 0.25]

# ---------------------------------------------------------
# 3. RANDOM DIGITS
# ---------------------------------------------------------
newsday_random_digits = [75, 77, 49, 55, 43, 79, 100, 16, 23, 33]

demand_random_digits = [68, 20, 15, 88, 98, 86, 73, 24, 52, 43]

# ---------------------------------------------------------
# 4. NEWSPAPER PARAMETERS
# ---------------------------------------------------------
purchase_cost = 0.23
selling_price = 0.40
salvage_value = 0.05
lost_profit_per_paper = 0.10

order_quantity = 65


# =========================================================
# FUNCTION TO CREATE RDA TABLE
# =========================================================
def create_rda_table(values, probs, column_name):

    cumulative_probs = []
    total = 0

    for p in probs:
        total += p
        cumulative_probs.append(round(total, 2))

    ranges = []
    start = 1

    for cp in cumulative_probs:

        end = round(cp * 100)

        if end == 100:
            ranges.append(f"{start:02d}-{end}")
        else:
            ranges.append(f"{start:02d}-{end:02d}")

        start = end + 1

    return pd.DataFrame({
        column_name: values,
        'Probability': probs,
        'Cumulative Probability': cumulative_probs,
        'Random Digit Range': ranges
    })


# =========================================================
# CREATE RDA TABLES
# =========================================================

newsday_table = create_rda_table(
    newsday_types,
    newsday_probs,
    'Type of Newsday'
)

good_table = create_rda_table(
    demand_levels,
    good_probs,
    'Demand'
)

fair_table = create_rda_table(
    demand_levels,
    fair_probs,
    'Demand'
)

poor_table = create_rda_table(
    demand_levels,
    poor_probs,
    'Demand'
)


# =========================================================
# FUNCTION TO GET VALUE FROM RANDOM DIGIT
# =========================================================
def get_value(random_digit, rda_table, column_name):

    for _, row in rda_table.iterrows():

        start_str, end_str = row['Random Digit Range'].split('-')

        start = int(start_str)
        end = int(end_str)

        if start <= random_digit <= end:
            return row[column_name]

    return None


# =========================================================
# SIMULATION
# =========================================================

newsday_results = []
demand_results = []
sales_results = []
unsold_results = []
revenue_results = []
salvage_results = []
purchase_results = []
lost_profit_results = []
daily_profit_results = []

for i in range(len(newsday_random_digits)):

    # -----------------------------------------------------
    # DETERMINE TYPE OF NEWSDAY
    # -----------------------------------------------------
    newsday = get_value(
        newsday_random_digits[i],
        newsday_table,
        'Type of Newsday'
    )

    newsday_results.append(newsday)

    # -----------------------------------------------------
    # DETERMINE DEMAND
    # -----------------------------------------------------
    if newsday == 'Good':

        demand = get_value(
            demand_random_digits[i],
            good_table,
            'Demand'
        )

    elif newsday == 'Fair':

        demand = get_value(
            demand_random_digits[i],
            fair_table,
            'Demand'
        )

    else:

        demand = get_value(
            demand_random_digits[i],
            poor_table,
            'Demand'
        )

    demand_results.append(demand)

    # -----------------------------------------------------
    # SALES
    # -----------------------------------------------------
    sales = min(order_quantity, demand)

    sales_results.append(sales)

    # -----------------------------------------------------
    # UNSOLD PAPERS
    # -----------------------------------------------------
    unsold = max(0, order_quantity - demand)

    unsold_results.append(unsold)

    # -----------------------------------------------------
    # REVENUE
    # -----------------------------------------------------
    revenue = sales * selling_price

    revenue_results.append(revenue)

    # -----------------------------------------------------
    # SALVAGE VALUE
    # -----------------------------------------------------
    salvage = unsold * salvage_value

    salvage_results.append(salvage)

    # -----------------------------------------------------
    # PURCHASE COST
    # -----------------------------------------------------
    purchase = order_quantity * purchase_cost

    purchase_results.append(purchase)

    # -----------------------------------------------------
    # LOST PROFIT
    # -----------------------------------------------------
    shortage = max(0, demand - order_quantity)

    lost_profit = shortage * lost_profit_per_paper

    lost_profit_results.append(lost_profit)

    # -----------------------------------------------------
    # DAILY PROFIT
    # -----------------------------------------------------
    daily_profit = revenue + salvage - purchase - lost_profit

    daily_profit_results.append(round(daily_profit, 2))


# =========================================================
# FINAL SIMULATION TABLE
# =========================================================
simulation_df = pd.DataFrame({

    'Day': range(1, 11),

    'Random Digit for Newsday': newsday_random_digits,

    'Type of Newsday': newsday_results,

    'Random Digit for Demand': demand_random_digits,

    'Demand': demand_results,

    'Order Quantity': [order_quantity] * 10,

    'Sales': sales_results,

    'Unsold Papers': unsold_results,

    'Revenue ($)': revenue_results,

    'Salvage Value ($)': salvage_results,

    'Purchase Cost ($)': purchase_results,

    'Lost Profit ($)': lost_profit_results,

    'Daily Profit ($)': daily_profit_results
})


# =========================================================
# PERFORMANCE MEASURES
# =========================================================
average_profit = round(simulation_df['Daily Profit ($)'].mean(), 2)

average_lost_profit = round(simulation_df['Lost Profit ($)'].mean(), 2)

average_demand = round(simulation_df['Demand'].mean(), 2)


# =========================================================
# DISPLAY OUTPUTS
# =========================================================

print("\nTYPE OF NEWSDAY RDA TABLE\n")
display(newsday_table)

print("\nGOOD NEWSDAY DEMAND TABLE\n")
display(good_table)

print("\nFAIR NEWSDAY DEMAND TABLE\n")
display(fair_table)

print("\nPOOR NEWSDAY DEMAND TABLE\n")
display(poor_table)

print("\nFINAL NEWSPAPER SIMULATION TABLE\n")
display(simulation_df)

print("\nPERFORMANCE MEASURES\n")

print(f"Average Daily Profit = ${average_profit}")

print(f"Average Lost Profit = ${average_lost_profit}")

print(f"Average Daily Demand = {average_demand}")


TYPE OF NEWSDAY RDA TABLE



,Type of Newsday,Probability,Cumulative Probability,Random Digit Range
0,Good,0.35,0.35,01-35
1,Fair,0.45,0.80,36-80
2,Poor,0.20,1.00,81-100



GOOD NEWSDAY DEMAND TABLE



,Demand,Probability,Cumulative Probability,Random Digit Range
0,40,0.25,0.25,01-25
1,60,0.40,0.65,26-65
2,80,0.35,1.00,66-100



FAIR NEWSDAY DEMAND TABLE



...


POOR NEWSDAY DEMAND TABLE



,Demand,Probability,Cumulative Probability,Random Digit Range
0,40,0.40,0.40,01-40
1,60,0.35,0.75,41-75
2,80,0.25,1.00,76-100



FINAL NEWSPAPER SIMULATION TABLE



,Day,Random Digit for Newsday,Type of Newsday,Random Digit for Demand,Demand,Order Quantity,Sales,Unsold Papers,Revenue ($),Salvage Value ($),Purchase Cost ($),Lost Profit ($),Daily Profit ($)
0,1,75,Fair,68,80,65,65,0,26.0,0.00,14.95,1.5,9.55
1,2,77,Fair,20,40,65,40,25,16.0,1.25,14.95,0.0,2.30
2,3,49,Fair,15,40,65,40,25,16.0,1.25,14.95,0.0,2.30
3,4,55,Fair,88,80,65,65,0,26.0,0.00,14.95,1.5,9.55
4,5,43,Fair,98,80,65,65,0,26.0,0.00,14.95,1.5,9.55
5,6,79,Fair,86,80,65,65,0,26.0,0.00,14.95,1.5,9.55
6,7,100,Poor,73,60,65,60,5,24.0,0.25,14.95,0.0,9.30
7,8,16,Good,24,40,65,40,25,16.0,1.25,14.95,0.0,2.30
8,9,23,Good,52,60,65,60,5,24.0,0.25,14.95,0.0,9.30
9,10,33,Good,43,60,65,60,5,24.0,0.25,14.95,0.0,9.30



PERFORMANCE MEASURES

Average Daily Profit = $7.3
Average Lost Profit = $0.6
Average Daily Demand = 62.0
